# Lund’s missing thermometer

If Lund’s weather station went offline, could two neighbouring stations estimate the temperature at the same moment? Use Malmö A and Skillinge A as inputs, hide Lund’s reading, then reveal it to score the prediction.

The simplest alternative is to copy Malmö. On 73 later timestamps the two-station model reduced mean absolute error from **0.474 °C to 0.401 °C**.

## 1. Load the observations

Run the notebook from its project folder to use the included CSV. On Colab, the same cell rebuilds it from three archived SMHI exports. It keeps quality flag **G** (checked and approved), joins by exact UTC timestamp and discards unmatched rows. The third export is named after Simrishamn, but the station in its header is **Skillinge A**.

In [ ]:
from io import StringIO
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

archive = "https://raw.githubusercontent.com/Rilr20/bth-matmod-proj/eba34d82a4bffd30b656fbc6a291e43253165f94/data/"
stations = {"lund_c": "smhi-lund.csv", "malmo_c": "smhi-malmo.csv",
            "skillinge_c": "smhi-simrishamn.csv"}

def read_smhi(filename):
    with urlopen(archive + filename, timeout=30) as response:
        lines = response.read().decode("utf-8-sig").splitlines()
    start = next(i for i, line in enumerate(lines) if line.startswith("Datum;Tid (UTC);"))
    rows = pd.read_csv(StringIO("\n".join(lines[start:])), sep=";", usecols=range(4))
    good = rows.loc[rows["Kvalitet"] == "G"].copy()
    good["time_utc"] = pd.to_datetime(good["Datum"] + " " + good["Tid (UTC)"], utc=True)
    return good.set_index("time_utc", verify_integrity=True)["Lufttemperatur"]

csv_path = Path("data/temperatures.csv")
if csv_path.is_file():
    data = pd.read_csv(csv_path, parse_dates=["time_utc"]).set_index("time_utc")
else:
    readings = {name: read_smhi(filename) for name, filename in stations.items()}
    data = pd.concat(readings, axis=1, join="inner").dropna()

data = data.sort_index()
print(f"{len(data)} matched timestamps")
data.head(3)

The first row is 2 August at 06:00 UTC. Malmö measured 15.9 °C and Skillinge 16.5 °C. These two numbers are the **features**. Lund measured 15.8 °C. That is the **target** the model learns to estimate.

## 2. Hide the later dates

August–October supplies 182 training examples. The 73 readings from 1 November through 7 December simulate a later outage. Splitting by time keeps observations from later in the same weather spell out of training when predicting earlier dates.

In [ ]:
features = ["malmo_c", "skillinge_c"]
train = data.loc[data.index < "2021-11-01"]
test = data.loc[data.index >= "2021-11-01"]

X_train, y_train = train[features], train["lund_c"]
X_test, y_test = test[features], test["lund_c"]
print(f"Training: {len(train)} | Test: {len(test)}")

## 3. Fit a two-input regression

During `.fit()`, scikit-learn chooses a weight per station and an offset to minimise squared errors on the training rows. During `.predict()`, it only sees the two neighbouring temperatures. The baseline simply copies Malmö at each test timestamp.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

estimated = model.predict(X_test)
baseline = X_test["malmo_c"]
print(f"Lund ≈ {model.intercept_:.3f} + {model.coef_[0]:.3f} × Malmö + {model.coef_[1]:.3f} × Skillinge")

## 4. Score the simulated outage

If the estimate is 5 °C and the reading is 7 °C, the absolute error is 2 °C. MAE averages those distances. RMSE gives larger misses more weight. Lower is better for both.

In [ ]:
scores = {}
for name, prediction in {"Copy Malmö": baseline, "Linear regression": estimated}.items():
    scores[name] = {
        "MAE (°C)": mean_absolute_error(y_test, prediction),
        "RMSE (°C)": root_mean_squared_error(y_test, prediction),
    }

results = pd.DataFrame(scores).T
results.round(3)

The model reduced MAE by about **0.07 °C** per reading, a **15.5%** reduction compared with copying Malmö. The baseline already works well, which makes the absolute gain modest.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6), layout="constrained")
ax.plot(test.index, y_test, color="#244643", linewidth=2, label="Measured in Lund")
ax.plot(test.index, estimated, color="#b15a32", linestyle="--", label="Two-station estimate")
ax.set(title="Lund during the simulated outage", ylabel="Air temperature (°C)", xlabel="Test period, 2021 (UTC)")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.2)
fig.autofmt_xdate()
plt.show()

In [ ]:
comparison = test.copy()
comparison["estimate_c"] = estimated
comparison["absolute_error_c"] = (comparison["lund_c"] - comparison["estimate_c"]).abs()
comparison.nlargest(3, "absolute_error_c").round(2)

The largest miss was 3 November at 06:00 UTC. Malmö measured 4.1 °C and Skillinge 6.7 °C. The model leaned towards Malmö and estimated 4.25 °C. Lund measured 6.1 °C.

## 5. Try your own readings

Change one temperature at a time. Which station affects the estimate more? The coefficients printed above explain the difference.

In [ ]:
example = pd.DataFrame({"malmo_c": [5.0], "skillinge_c": [7.0]})
print(f"Estimated Lund temperature: {model.predict(example)[0]:.1f} °C")

## What this test covers

Both neighbouring stations have to report at the moment of the outage. The test covers about five weeks at 06:00 and 18:00 UTC. To assess summer heatwaves or midday outages, the next experiment would need more seasons and observations at more hours.

**Data credit.** SMHI air-temperature observations, filtered to quality flag G and aligned by timestamp. [Source snapshot](https://github.com/Rilr20/bth-matmod-proj/tree/eba34d82a4bffd30b656fbc6a291e43253165f94/data) · [SMHI terms](https://www.smhi.se/data/om-smhis-data/villkor-for-anvandning) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).